# Datos totales. Viendo qué filtramos

In [1]:
from datasets import load_dataset
import pandas as pd

# 1. Cargar el dataset CLINC150 (usamos la versión 'plus' que tiene todos los datos)
print("Cargando el dataset...")
dataset = load_dataset("clinc_oos", 'plus')
dataset

Cargando el dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text', 'intent'],
        num_rows: 15250
    })
    validation: Dataset({
        features: ['text', 'intent'],
        num_rows: 3100
    })
    test: Dataset({
        features: ['text', 'intent'],
        num_rows: 5500
    })
})

Vale el dataset viene partido en 3, pero tenemos pocos datos. Opciones:
1. Juntamos los tres datos y la forma de validar sería haciendo validación cruzada (obtenemos por tanto una proyección del rendimiento del modelo y no una evaluación en sí misma), pero tenemos más datos para entrenar en total
2. Juntamos dos de tres conjuntos de datos (diría que los que tengan más datos, o sea train + test/val el más abundante), hacemos validación cruzada para optimización de hiperparámetros sobre esos datos combinados, y sobre el restante es nuestra métrica de evaluación real. Pero tendríamos menos datos para entrenar.

Optamos por 2.

In [2]:
df_train = dataset['train'].to_pandas()
df_val = dataset['validation'].to_pandas()
df_test = dataset['test'].to_pandas()

Veamos las intenciones que hay.

In [3]:
nombres_intenciones = dataset['train'].features['intent'].names
nombres_intenciones[:20]

['restaurant_reviews',
 'nutrition_info',
 'account_blocked',
 'oil_change_how',
 'time',
 'weather',
 'redeem_rewards',
 'interest_rate',
 'gas_type',
 'accept_reservations',
 'smart_home',
 'user_name',
 'report_lost_card',
 'repeat',
 'whisper_mode',
 'what_are_your_hobbies',
 'order',
 'jump_start',
 'schedule_meeting',
 'meeting_schedule']

In [4]:
# Filtramos las de cocina (Gemini)
intenciones_cocina = [
    'ingredient_substitution', # Sustitución de ingredientes
    'recipe',                  # Búsqueda de recetas
    'cook_time',               # Tiempo de cocción
    'nutrition_info',          # Información nutricional
    'calories',                # Calorías
    'meal_suggestion',         # Sugerencia de comida
    'ingredients_list',        # Ingredientes de una receta
    'food_last',               # Caducidad

]

Filtramos datasets.

In [5]:
# Creamos una nueva columna con el nombre de la intención en texto
df_train['nombre_intencion'] = df_train['intent'].apply(lambda x: nombres_intenciones[x])
df_val['nombre_intencion'] = df_val['intent'].apply(lambda x: nombres_intenciones[x])
df_test['nombre_intencion'] = df_test['intent'].apply(lambda x: nombres_intenciones[x])

# Filtramos. Nos quedamos solo con las filas cuya intención esté en nuestra lista
cocina_train = df_train[df_train['nombre_intencion'].isin(intenciones_cocina)]
cocina_val = df_val[df_val['nombre_intencion'].isin(intenciones_cocina)]
cocina_test = df_test[df_test['nombre_intencion'].isin(intenciones_cocina)]

# Resultados
print("Train")
print(f"Total de ejemplos originales: {len(df_train)}")
print(f"Total de ejemplos de cocina: {len(cocina_train)}")

print("\nVal")
print(f"Total de ejemplos originales: {len(df_val)}")
print(f"Total de ejemplos de cocina: {len(cocina_val)}")

print("\nTest")
print(f"Total de ejemplos originales: {len(df_test)}")
print(f"Total de ejemplos de cocina: {len(cocina_test)}")

Train
Total de ejemplos originales: 15250
Total de ejemplos de cocina: 800

Val
Total de ejemplos originales: 3100
Total de ejemplos de cocina: 160

Test
Total de ejemplos originales: 5500
Total de ejemplos de cocina: 240


## Dataset de cocina. Balanceo de clases

In [6]:
cocina_train

,text,intent,nombre_intencion
5500,what's the nutritional info for spaghetti,1,nutrition_info
5501,what's the nutritional info for pizza,1,nutrition_info
5502,how healthy are potato skins,1,nutrition_info
5503,how healthy is spaghetti,1,nutrition_info
5504,share the nutrition info for brownies with me,1,nutrition_info
...,...,...,...
14495,can you instruct me on how to make german choc...,104,recipe
14496,how do i make the perfect omelette,104,recipe
14497,i want to make sour dough bread please find a ...,104,recipe
14498,i need a really good recipe for making doughnuts,104,recipe


In [7]:
def comprobar_balanceo(dict_datasets):
    """
    dict_datasets: Diccionario con formato {'Nombre': dataframe}
    Ejemplo: {'Train': cocina_train, 'Val': cocina_val, 'Test': cocina_test}
    """
    conteos = []

    for nombre, df in dict_datasets.items():
        # Obtenemos el conteo y lo convertimos a DataFrame
        c = df['nombre_intencion'].value_counts().to_frame()
        c.columns = [nombre] # Renombramos la columna con el nombre del set
        conteos.append(c)

    # Unimos todos los conteos por el índice (nombre_intencion)
    resumen = pd.concat(conteos, axis=1).fillna(0).astype(int)

    # Añadimos una columna de 'Total' para ver el peso global de cada clase
    resumen['Total'] = resumen.sum(axis=1)

    return resumen.sort_values(by='Total', ascending=False)

In [8]:
# Metemos tus 3 datasets en un diccionario
mis_datasets = {
    'Train': cocina_train,
    'Val': cocina_val,
    'Test': cocina_test
}

# Ejecutamos la comprobación
df_balanceo = comprobar_balanceo(mis_datasets)
print(df_balanceo)

                         Train  Val  Test  Total
nombre_intencion                                
nutrition_info             100   20    30    150
food_last                  100   20    30    150
cook_time                  100   20    30    150
ingredient_substitution    100   20    30    150
calories                   100   20    30    150
ingredients_list           100   20    30    150
meal_suggestion            100   20    30    150
recipe                     100   20    30    150


Tenemos 8 clases perfectamente balanceadas.

## Combinando train + test. Apartamos val

In [9]:
dataset = pd.concat([cocina_train, cocina_test])

Veamos algunas muestras de cada clase.

In [10]:
N = 30 # Número de ejemplos

for intencion in intenciones_cocina:
    print(f"==== Intención: {intencion} ====")

    # Filtramos el dataset para obtener solo las filas de la intención actual
    datos_filtrados = dataset[dataset['nombre_intencion'] == intencion]

    # Tomamos los primeros N registros
    primeros = datos_filtrados.head(N)

    # Iteramos sobre esos 10 registros para imprimir el texto
    for index, fila in primeros.iterrows():
        print(f"  {fila['text']}")
    print("")

==== Intención: ingredient_substitution ====
  what about changing the sugar for baking soda
  can i swap sugar for salt
  would it be possible to replace the salt with baking soda
  instead of pepper, can i use salt
  can i take out the olive oil and use lard
  can i use water instead of milk
  it is okay to use water in place of milk
  it is okay to replace water instead of milk
  is it okay to switch water for milk
  will it be okay if i use water instead of milk
  can i substitute ginger for garlic
  can i substitute rice for potatoes
  can i substitute oregano for basil
  can i substitute bacon for sausage
  can i substitute honey for sugar
  can i switch cream with milk in a recipe
  can i exchange milk for cream in recipes
  is milk an ok substitute for cream
  can i use milk instead of cream
  can i switch cream out for milk
  can i substitute skim milk for whole milk
  is it okay to substitute oregano for basil
  can i use colby jack cheese instead of cheddar in a recipe
  can